# Deep Convolutional GAN

---

## 一、Discriminator

In [1]:
import torch
import torch.nn as nn

class Discriminator(nn.Module):
    def __init__(self, img_channels, features_disc):
        super(Discriminator, self).__init__()
        """
        Conv2D:
            (batch, img_channels,    64, 64) -> (batch, features_disc, 32, 32)
        ConvBlocks:
            (batch, features_disc,   32, 32) -> (batch, features_disc*2, 16, 16)
            (batch, features_disc*2, 16, 16) -> (batch, features_disc*4,  8,  8)
            (batch, features_disc*4,  8,  8) -> (batch, features_disc*8,  4,  4)
            (batch, features_disc*8,  4,  4) -> (batch, 1, 1, 1), 即输出一个用于鉴别的0/1单值
        """
        self.disc = nn.Sequential(
            nn.Conv2d(
                in_channels=img_channels,
                out_channels=features_disc,
                kernel_size=4,
                stride=2,
                padding=1,
            ),
            nn.LeakyReLU(0.2),
            self.conv_block(features_disc, features_disc*2, kernel_size=4, stride=2, padding=1),
            self.conv_block(features_disc*2, features_disc*4, kernel_size=4, stride=2, padding=1),
            self.conv_block(features_disc*4, features_disc*8, kernel_size=4, stride=2, padding=1),
            self.conv_block(features_disc*8, 1, kernel_size=4, stride=2, padding=0),
            nn.Sigmoid(), # in (0, 1)
        )
        
    def conv_block(self, in_channels, out_channels, kernel_size, stride, padding):
        return nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size,
                stride,
                padding,
                bias=False,
            ),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.2),
        )
        
    def forward(self, x):
        return self.disc(x)

---

## 二、Generator

In [2]:
class Generator(nn.Module):
    def __init__(self, z_dim, img_channels, features_gen):
        super(Generator, self).__init__()
        """
        ConvBlocks:
            (batch, z_dim, 1, 1) -> (batch, features_gen*16, 4, 4)
            (batch, features_gen*16, 4, 4) -> (batch, features_gen*8, 8, 8)
            (batch, features_gen*8, 8, 8) -> (batch, features_gen*4, 16, 16)
            (batch, features_gen*4, 16, 16) -> (batch, features_gen*2, 32, 32)
        ConvTranspose2d:
            (batch, features_gen*2, 32, 32) -> (batch, img_channels, 64, 64)
        """
        self.gen = nn.Sequential(
            self.conv_block(z_dim, features_gen*16, 4, 1, 0),
            self.conv_block(features_gen*16, features_gen*8, 4, 2, 1),
            self.conv_block(features_gen*8,  features_gen*4, 4, 2, 1),
            self.conv_block(features_gen*4,  features_gen*2, 4, 2, 1),
            nn.ConvTranspose2d(
                features_gen*2, img_channels, kernel_size=4, stride=2, padding=1,
            ),
            nn.Tanh() # in normalized [-1, 1]
        )
    
    def conv_block(self, in_channels, out_channels, kernel_size, stride, padding):
        """
        ConvTranspose2D: 反卷积操作, 与卷积操作的效果相反
        反卷积操作是一种上采样操作, 会放大特征图尺寸, 常用于图像生成任务
        """
        return nn.Sequential(
            nn.ConvTranspose2d(
                in_channels,
                out_channels,
                kernel_size,
                stride,
                padding,
                bias=False,
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
        )
        
    def forward(self, x):
        return self.gen(x)

---

## 三、Training

In [3]:
import torch.optim as optim
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

"""
设置所有的超参数
"""
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LEARNING_RATE = 2e-4
BATCH_SIZE = 128
IMG_SIZE = 64
IMG_CHANNELS = 3
Z_DIM = 100
NUM_EPOCHS = 5
FEATURES_DISC = 64
FEATURES_GEN = 64

transforms = transforms.Compose(
    [
        transforms.Resize(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(
            (0.5, 0.5, 0.5),
            (0.5, 0.5, 0.5),
        ),
    ]
)

dataset = datasets.CIFAR10(root="/mnt/ssd2/wangle/datasets/cifar10", train=True, transform=transforms, download=False)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
len(dataset)

50000

In [4]:
gen = Generator(Z_DIM, IMG_CHANNELS, FEATURES_GEN).to(device)
gen.train()
disc = Discriminator(IMG_CHANNELS, FEATURES_DISC).to(device)
disc.train()

opt_gen = torch.optim.Adam(gen.parameters(), lr=LEARNING_RATE, betas=(0.5, 0.999))
opt_disc = torch.optim.Adam(disc.parameters(), lr=LEARNING_RATE, betas=(0.5, 0.999))
criterion = nn.BCELoss()

fixed_noise = torch.randn(1, Z_DIM, 1, 1).to(device)

for epoch in range(NUM_EPOCHS):
    for batch_idx, (real, _) in enumerate(dataloader):
        real = real.to(device)
        noise = torch.randn(BATCH_SIZE, Z_DIM, 1, 1).to(device)
        fake = gen(noise) # (batch, 3, 64, 64)
        
        """
        train Discriminator: maxmize log(D(x)) + log(1 - D(G(z)))
        """
        disc_real = disc(real).reshape(-1) # (batch, 1, 1, 1) -> (batch, )
        loss_disc_real = criterion(disc_real, torch.ones_like(disc_real))
        disc_fake = disc(fake).reshape(-1) # (batch, 1, 1, 1) -> (batch, )
        loss_disc_fake = criterion(disc_fake, torch.zeros_like(disc_fake))
        loss_disc = (loss_disc_real + loss_disc_fake) / 2
        disc.zero_grad()
        loss_disc.backward(retain_graph=True)
        opt_disc.step()
    
        """
        train Generator: maxmize log(D(G(z)))
        """
        disc_fake = disc(fake).reshape(-1) # (batch, 1, 1, 1) -> (batch, )
        loss_gen = criterion(disc_fake, torch.ones_like(disc_fake))
        gen.zero_grad()
        loss_gen.backward()
        opt_gen.step()